# Task 3 Demo - Anomaly Detection Methods and Top-50 Selection

In this notebook I demonstrate anomaly detection with multiple methods and deterministic top-50 output.

What I show:

1. Build anomaly-focused features.
2. Compare Isolation Forest, LOF, and distance-based kNN scoring.
3. Analyze overlap between methods.
4. Export exactly 50 anomaly IDs in assignment format.

In [1]:
from pathlib import Path

import pandas as pd
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor, NearestNeighbors

from data_mining_assignment.core.data_io import ArticleDataset
from data_mining_assignment.tasks.preprocessing import TextNormalizer, TextPreprocessor

## Load data and build anomaly feature spaces

In [2]:
project_root_path = Path.cwd().parent
results_data_directory_path = project_root_path / "data" / "results"
results_data_directory_path.mkdir(parents=True, exist_ok=True)

articles_csv_path = project_root_path / "data" / "raw" / "articles.csv"
articles_data_frame = ArticleDataset(input_csv_path=articles_csv_path).load_articles()

document_ids = articles_data_frame["doc_id"].tolist()
raw_texts = articles_data_frame["text"].tolist()

text_normalizer = TextNormalizer()
normalized_bundle = text_normalizer.normalize_for_both_tasks(raw_texts)

anomaly_preprocessor = TextPreprocessor(
    vectorization_model_name="tfidf_lsa_dense",
    max_features=30000,
    min_document_frequency=1,
    max_document_frequency=1.0,
    ngram_range=(3, 5),
    analyzer_mode="char_wb",
    dense_embedding_dimension=256,
    random_seed=42,
)
anomaly_feature_matrix = anomaly_preprocessor.fit_transform(normalized_bundle.anomaly_texts)

anomaly_feature_matrix.shape

(2164, 256)

## Method 1 - Isolation Forest

In [3]:
isolation_forest_model = IsolationForest(contamination=0.02, random_state=42)
isolation_forest_model.fit(anomaly_feature_matrix)

# Lower score means more anomalous for score_samples.
isolation_forest_scores = isolation_forest_model.score_samples(anomaly_feature_matrix)

isolation_forest_ranked = (
    pd.DataFrame({"doc_id": document_ids, "score": isolation_forest_scores})
    .sort_values("score", ascending=True)
    .reset_index(drop=True)
)
isolation_forest_ranked["rank"] = isolation_forest_ranked.index + 1
isolation_forest_ranked.to_csv(results_data_directory_path / "notebook_07_isolation_forest_ranking.csv", index=False)

isolation_forest_ranked.head(20)

,doc_id,score,rank
0,DOC_00909,-0.519633,1
1,DOC_01750,-0.496353,2
2,DOC_01025,-0.492676,3
3,DOC_01622,-0.488615,4
4,DOC_01211,-0.488554,5
5,DOC_01114,-0.486392,6
6,DOC_00031,-0.485103,7
7,DOC_01449,-0.482385,8
8,DOC_02026,-0.479044,9
9,DOC_02075,-0.478267,10


## Method 2 - Local Outlier Factor (LOF)

In [4]:
lof_model = LocalOutlierFactor(n_neighbors=35, contamination=0.02)
lof_predictions = lof_model.fit_predict(anomaly_feature_matrix)

# LOF stores negative factors; lower (more negative) means more anomalous.
lof_raw_factors = lof_model.negative_outlier_factor_
lof_ranked = pd.DataFrame({"doc_id": document_ids, "score": lof_raw_factors, "predicted_outlier": lof_predictions == -1})
lof_ranked = lof_ranked.sort_values("score", ascending=True).reset_index(drop=True)
lof_ranked["rank"] = lof_ranked.index + 1
lof_ranked.to_csv(results_data_directory_path / "notebook_07_lof_ranking.csv", index=False)

lof_ranked.head(20)

,doc_id,score,predicted_outlier,rank
0,DOC_00098,-1.428581,True,1
1,DOC_00089,-1.423546,True,2
2,DOC_00155,-1.402581,True,3
3,DOC_00464,-1.377624,True,4
4,DOC_00030,-1.342073,True,5
5,DOC_00001,-1.333763,True,6
6,DOC_00711,-1.325969,True,7
7,DOC_01449,-1.325656,True,8
8,DOC_01894,-1.317507,True,9
9,DOC_00626,-1.311419,True,10


## Method 3 - Distance-based kNN score

In [5]:
neighbor_model = NearestNeighbors(n_neighbors=20, metric="cosine")
neighbor_model.fit(anomaly_feature_matrix)

neighbor_distances, _ = neighbor_model.kneighbors(anomaly_feature_matrix)
# Higher average distance means less similar neighborhood.
knn_distance_scores = neighbor_distances.mean(axis=1)

knn_ranked = (
    pd.DataFrame({"doc_id": document_ids, "score": knn_distance_scores})
    .sort_values("score", ascending=False)
    .reset_index(drop=True)
)
knn_ranked["rank"] = knn_ranked.index + 1
knn_ranked.to_csv(results_data_directory_path / "notebook_07_knn_distance_ranking.csv", index=False)

knn_ranked.head(20)

,doc_id,score,rank
0,DOC_00476,0.687164,1
1,DOC_01075,0.685802,2
2,DOC_02022,0.684306,3
3,DOC_00572,0.680118,4
4,DOC_00510,0.678034,5
5,DOC_00374,0.677394,6
6,DOC_00107,0.676349,7
7,DOC_01994,0.676306,8
8,DOC_01621,0.675687,9
9,DOC_01472,0.674525,10


## Compare overlaps between top-50 sets

In [6]:
def top_k_ids(ranked_table: pd.DataFrame, top_k: int = 50) -> set[str]:
    """Returns top-k ids from a ranked anomaly table."""
    return set(ranked_table.head(top_k)["doc_id"].astype(str).tolist())


if_top_50 = top_k_ids(isolation_forest_ranked, 50)
lof_top_50 = top_k_ids(lof_ranked, 50)
knn_top_50 = top_k_ids(knn_ranked, 50)

overlap_table = pd.DataFrame(
    [
        {"pair": "if_vs_lof", "overlap": len(if_top_50.intersection(lof_top_50))},
        {"pair": "if_vs_knn", "overlap": len(if_top_50.intersection(knn_top_50))},
        {"pair": "lof_vs_knn", "overlap": len(lof_top_50.intersection(knn_top_50))},
    ]
)
overlap_table.to_csv(results_data_directory_path / "notebook_07_top50_overlap_table.csv", index=False)
overlap_table

,pair,overlap
0,if_vs_lof,20
1,if_vs_knn,2
2,lof_vs_knn,3


## Export assignment-formatted anomalies.csv (exactly 50 ids)

In [7]:
final_top_50 = isolation_forest_ranked.head(50).copy()
final_top_50["anomaly"] = final_top_50.index + 1
anomaly_submission_data_frame = final_top_50[["anomaly", "doc_id"]]

# This file matches the assignment output format.
anomaly_submission_data_frame.to_csv(results_data_directory_path / "anomalies.csv", index=False)
anomaly_submission_data_frame.to_csv(results_data_directory_path / "notebook_07_anomalies_candidate.csv", index=False)

anomaly_submission_data_frame.head()


,anomaly,doc_id
0,1,DOC_00909
1,2,DOC_01750
2,3,DOC_01025
3,4,DOC_01622
4,5,DOC_01211
